In [2]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("Olist-Silver") \
    .master("local[*]") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 13:32:47 WARN Utils: Your hostname, MacBook-Air-de-Sarah.local, resolves to a loopback address: 127.0.0.1; using 10.26.1.25 instead (on interface en0)
26/06/17 13:32:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 13:32:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
base_path = "/Users/sarah/Desktop/brief/BigData"

orders = spark.read.parquet(f"{base_path}/data/1_bronze/orders")
payments = spark.read.parquet(f"{base_path}/data/1_bronze/payments")
reviews = spark.read.parquet(f"{base_path}/data/1_bronze/reviews")

print(f"orders : {orders.count()} rows")
print(f"payments : {payments.count()} rows")
print(f"reviews : {reviews.count()} rows")


orders : 99441 rows
payments : 103886 rows
reviews : 99224 rows


In [5]:
for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews)]:
    print(f"\n{name} :")
    for col in df.columns:
        null_count = df.filter(df[col].isNull()).count()
        if null_count > 0:
            print(f"  {col} : {null_count} nulls")


orders :
  order_approved_at : 160 nulls
  order_delivered_carrier_date : 1783 nulls
  order_delivered_customer_date : 2965 nulls

payments :

reviews :
  review_comment_title : 87656 nulls
  review_comment_message : 58247 nulls


In [6]:

for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews)]:
    total = df.count()
    distinct = df.distinct().count()
    duplicates = total - distinct
    print(f"{name} : {duplicates} duplicates")

orders : 0 duplicates
payments : 0 duplicates
reviews : 0 duplicates


In [7]:
from pyspark.sql.functions import count

reviews.groupBy("review_id").count().filter("count > 1").show(truncate=False)

+--------------------------------+-----+
|review_id                       |count|
+--------------------------------+-----+
|f144ac1998474653203c861be02fd31f|2    |
|e4f5fbbbf2fa8f259e02e0e529eeae26|2    |
|868c4827b9ab7272338ae1c95d192505|2    |
|a8ffb30c74397c7cf37ab595d3c1662e|2    |
|f6e05a3f6dbc3bdb1ec3068725c67677|2    |
|c9a406d3076559e35f713ec82223d39c|2    |
|95cfb15ad3477fec574b561d1daea972|2    |
|dbcd301b59c7e85e61c10afd146e68ad|2    |
|9a07cb8136a1792e7984d827e2336d60|2    |
|dd43a798c9e8baf7b69297bb076c517d|2    |
|ef60daa08e71f279bffca277370f205f|2    |
|4812973f7fb59a488cb5fcbafe275581|2    |
|f45e3373e177f56944a549ff601f838b|2    |
|9d7b240dff5539983f5def389e2c1bf7|2    |
|4e9a141db068cf6856ce85ce8c3ef682|2    |
|c1dd7c6a2154caaf1a6accd68842286f|2    |
|bee3c76b6017ec9deaeef78bd5dbef21|2    |
|1961cafe1d1cbd4cc701c2f0160467ff|2    |
|d8a302244615111a0399d88db9916fd2|2    |
|fde5986d35c89aa1b6ce4149de82a0d3|2    |
+--------------------------------+-----+
only showing top